In [2]:
"""
扫描项目文件结构 + 审计蛋白嵌入文件
用法: python scan_embeddings.py [项目根目录] [--max-depth 4]
依赖: pip install h5py numpy (torch/pandas/safetensors 可选, 按需安装)
"""
import os, sys, argparse, pickle, warnings
from pathlib import Path
from collections import defaultdict

# ---------- 可选依赖探测 ----------
def _try_import(name):
    try:
        return __import__(name)
    except ImportError:
        return None

h5py    = _try_import("h5py")
np      = _try_import("numpy")
torch   = _try_import("torch")
pandas  = _try_import("pandas")
st      = _try_import("safetensors")

EMBED_EXTS = {".h5", ".hdf5", ".npy", ".npz", ".pt", ".pth", ".pkl", ".pickle", ".safetensors"}
EXPECTED_DIMS = {650: 1280, 1024: 1024, 2560: 2560, 5120: 5120}  # 模型参数量 -> 嵌入维度, 可自行扩充

# ---------- 1. 目录树 ----------
def print_tree(root: Path, max_depth=4, indent=""):
    """打印目录树, 标注 [EMB] 的为嵌入文件"""
    if max_depth < 0:
        return
    try:
        entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    except PermissionError:
        return
    for i, p in enumerate(entries):
        last = (i == len(entries) - 1)
        prefix = indent + ("└── " if last else "├── ")
        if p.is_dir():
            print(f"{prefix}{p.name}/")
            print_tree(p, max_depth - 1, indent + ("    " if last else "│   "))
        else:
            tag = " [EMB]" if p.suffix.lower() in EMBED_EXTS else ""
            size = _human_size(p.stat().st_size)
            print(f"{prefix}{p.name} ({size}){tag}")

def _human_size(n):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024

# ---------- 2. 各格式审计器 ----------
def audit_h5(path: Path):
    """HDF5: 递归遍历组/数据集, 报告形状/dtype/属性"""
    if h5py is None:
        return ["⚠ h5py 未安装, 无法读取 (pip install h5py)"]
    issues, info = [], []
    with h5py.File(path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                info.append(f"  数据集 '{name}': shape={obj.shape}, dtype={obj.dtype}")
                # 蛋白嵌入专用检查
                if obj.ndim == 2 and obj.shape[1] in (1024, 1280, 2560, 5120):
                    info.append(f"    -> 疑似逐残基嵌入池化后的 (L, dim) 或 (N_protein, dim) 矩阵")
                if obj.ndim == 3:
                    info.append(f"    -> 逐残基嵌入 (N_seq, L, dim), 需确认 L 含 padding!")
                if obj.dtype not in (np.float32, np.float16) if np else False:
                    issues.append(f"  ⚠ '{name}' dtype={obj.dtype}, 建议 float32/float16 节省空间")
            elif isinstance(obj, h5py.Group):
                n = len(obj.keys())
                info.append(f"  组 '{name}/' ({n} 个子对象)")
                # 若组内是每序列一个数据集 (常见 ESM 提取格式)
                if n > 0 and n < 200:
                    first = list(obj.keys())[0]
                    d = obj[first]
                    if isinstance(d, h5py.Dataset):
                        info.append(f"    样本键: '{first}', shape={d.shape}")
                        # ID 键命名一致性
                        for k in list(obj.keys())[:5]:
                            if not k.replace("_", "").replace(".", "").isalnum():
                                issues.append(f"  ⚠ 键 '{k}' 含特殊字符, 可能导致下游索引失败")
        f.visititems(visit)
        # 顶层属性 (常存元数据: 模型名/层号/池化方式)
        for k, v in f.attrs.items():
            info.append(f"  根属性: {k} = {v}")
        if not any("模型" in str(v) or "model" in str(v) or "layer" in str(v) for v in f.attrs.values()):
            issues.append("  ⚠ 无模型名/层号元数据, 无法追溯嵌入来源")
    return info, issues

def audit_numpy(path: Path):
    if np is None:
        return [], ["⚠ numpy 未安装"]
    issues, info = [], []
    if path.suffix == ".npy":
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        info.append(f"  ndarray: shape={arr.shape}, dtype={arr.dtype}")
        _check_emb_shape(arr.shape, issues)
    else:  # .npz
        z = np.load(path, mmap_mode="r")
        for k in z.files:
            arr = z[k]
            info.append(f"  键 '{k}': shape={arr.shape}, dtype={arr.dtype}")
            _check_emb_shape(arr.shape, issues)
        if "ids" not in z.files and "keys" not in z.files and "names" not in z.files:
            issues.append("  ⚠ .npz 中无 ids/keys 字段, 序列与嵌入的对应关系将丢失!")
    return info, issues

def _check_emb_shape(shape, issues):
    """蛋白嵌入形状启发式检查"""
    if len(shape) == 1 and shape[0] not in (1024, 1280, 2560, 5120, 256, 320, 384, 480, 640):
        pass  # 单条序列, 难判断
    if len(shape) == 3:
        n, L, d = shape
        if d not in EXPECTED_DIMS.values():
            issues.append(f"  ⚠ 第3维={d}, 不在常见 PLM 嵌入维度 {sorted(EXPECTED_DIMS.values())} 中")
        if L > 1100:
            issues.append(f"  ⚠ 序列长度 L={L} 超过 ESM-2 上限 1022, 检查是否滑窗/截断")
        if L < 50:
            issues.append(f"  ⚠ L={L} 异常短, 若多条序列可能被 padding 后统一截断, 检查掩码!")
    if len(shape) == 2 and shape[-1] in (1024, 1280, 2560, 5120):
        info_ = f"  -> 形如 (N, dim): 疑似蛋白级池化嵌入"
    # 逐残基嵌入若被提前池化成 (N, dim), 1D 卷积自编码器将失去位置信息 — 见文末检查清单

def audit_torch(path: Path):
    if torch is None:
        return [], ["⚠ torch 未安装, 无法读取 .pt"]
    issues, info = [], []
    obj = torch.load(path, map_location="cpu", weights_only=False)
    def describe(o, prefix=""):
        if isinstance(o, dict):
            for k, v in list(o.items())[:10]:
                if isinstance(v, torch.Tensor):
                    info.append(f"  {prefix}'{k}': tensor shape={tuple(v.shape)}, dtype={v.dtype}")
                    _check_emb_shape(tuple(v.shape), issues)
                else:
                    info.append(f"  {prefix}'{k}': {type(v).__name__}")
                    describe(v, prefix + "  ")
        elif isinstance(o, torch.Tensor):
            info.append(f"  tensor: shape={tuple(o.shape)}, dtype={o.dtype}")
        else:
            info.append(f"  {type(o).__name__}: {str(o)[:100]}")
    describe(obj)
    return info, issues

def audit_pickle(path: Path):
    issues, info = [], []
    warnings.filterwarnings("ignore")
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
    except Exception as e:
        return [f"  ✗ 无法反序列化: {e}"], []
    if isinstance(obj, dict):
        keys = list(obj.keys())
        info.append(f"  dict, {len(keys)} 个键; 样本: {keys[:5]}")
        v = obj[keys[0]]
        if np is not None and isinstance(v, np.ndarray):
            info.append(f"  值样本: shape={v.shape}, dtype={v.dtype}")
            _check_emb_shape(v.shape, issues)
        if all(isinstance(k, str) and "|" in k for k in keys[:5]):
            info.append("  -> 键含 '|', 疑似 PPI 对 ID (proteinA|proteinB)")
    elif isinstance(obj, (list, tuple)) and len(obj) == 2:
        info.append(f"  (ids, embeddings) 元组: {len(obj[0])} 条, 嵌入 shape={np.asarray(obj[1]).shape if np else '?'}")
    else:
        info.append(f"  {type(obj).__name__}, len={len(obj) if hasattr(obj, '__len__') else '?'}")
    return info, issues

def audit_safetensors(path: Path):
    if st is None:
        return [], ["⚠ safetensors 未安装"]
    from safetensors import safe_open
    issues, info = [], []
    with safe_open(path, framework="numpy") as f:
        for k in f.keys()[:20]:
            t = f.get_slice(k)
            info.append(f"  '{k}': shape={t.get_shape()}, dtype={t.get_dtype()}")
    return info, issues

# ---------- 3. 主流程 ----------
AUDITORS = {
    ".h5": audit_h5, ".hdf5": audit_h5,
    ".npy": audit_numpy, ".npz": audit_numpy,
    ".pt": audit_torch, ".pth": audit_torch,
    ".pkl": audit_pickle, ".pickle": audit_pickle,
    ".safetensors": audit_safetensors,
}

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("root", nargs="?", default=".", help="项目根目录")
    ap.add_argument("--max-depth", type=int, default=4, help="目录树最大深度")
    ap.add_argument("--skip-audit", action="store_true", help="只打印目录树, 不打开嵌入文件")
    args, _ = ap.parse_known_args()   # 忽略 Jupyter 注入的 --f=... 参数

    root = Path(args.root).resolve()

    print(f"{'='*70}\n项目结构: {root}\n{'='*70}")
    print_tree(root, args.max_depth)

    # 收集嵌入文件
    emb_files = [p for p in root.rglob("*")
                 if p.is_file() and p.suffix.lower() in EMBED_EXTS
                 and ".git" not in p.parts and "__pycache__" not in p.parts]

    if args.skip_audit or not emb_files:
        print(f"\n嵌入文件: {len(emb_files)} 个 (审计{'已跳过' if args.skip_audit else '未执行'})")
        return

    print(f"\n{'='*70}\n嵌入文件审计 ({len(emb_files)} 个)\n{'='*70}")
    stats = defaultdict(lambda: {"count": 0, "total_MB": 0.0, "issues": 0})
    for p in emb_files:
        size_mb = p.stat().st_size / 1024 / 1024
        stats[p.suffix]["count"] += 1
        stats[p.suffix]["total_MB"] += size_mb
        print(f"\n◆ {p.relative_to(root)}  ({_human_size(p.stat().st_size)})")
        auditor = AUDITORS.get(p.suffix.lower())
        try:
            result = auditor(p) if auditor else ([], [f"  ⚠ 无审计器 ({p.suffix})"])
            info, issues = result if isinstance(result, tuple) else (result, [])
            for line in info[:30]:   print(line)
            for line in issues:      print(line); stats[p.suffix]["issues"] += 1
            if len(info) > 30:
                print(f"  ... 其余 {len(info)-30} 条省略")
            if not issues:
                print("  ✓ 未发现结构性问题")
        except Exception as e:
            print(f"  ✗ 读取失败: {type(e).__name__}: {e}")
            stats[p.suffix]["issues"] += 1

    # 汇总
    print(f"\n{'='*70}\n汇总\n{'='*70}")
    print(f"{'格式':<15}{'数量':>6}{'总大小':>12}{'问题数':>8}")
    for ext, s in sorted(stats.items()):
        print(f"{ext:<15}{s['count']:>6}{s['total_MB']:>10.1f}MB{s['issues']:>8}")
    total_issues = sum(s["issues"] for s in stats.values())
    if total_issues:
        print(f"\n⚠ 共 {total_issues} 个问题, 重点处理 '⚠' 标记项后再进入模型训练")

if __name__ == "__main__":
    main()


项目结构: D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)
├── .vscode/
│   └── settings.json (204.0B)
├── checkpoints/
│   ├── 2d_selfattention_best.pt (2.0MB) [EMB]
│   └── 2d_selfattention_best_maxreadout_backup.pt (2.0MB) [EMB]
├── data/
│   ├── __pycache__/
│   │   └── data.cpython-311.pyc (9.8KB)
│   ├── distmaps/
│   │   ├── Crossattention_2V8S_Q14677_Q9UEU0.png (289.3KB)
│   │   ├── DSCRIPT-like_1B34_P62314_P62316.png (228.3KB)
│   │   ├── DSCRIPT-like_2A24_P27540_Q99814.png (313.7KB)
│   │   ├── DSCRIPT-like_5F5S_P55081_Q8NAV1.png (287.8KB)
│   │   ├── Selfattention_1B34_P62314_P62316.png (227.5KB)
│   │   ├── Selfattention_1F3V_Q12933_Q15628.png (382.7KB)
│   │   ├── Selfattention_2V8S_Q14677_Q9UEU0.png (295.1KB)
│   │   ├── Selfattention_3F1S_P22891_Q9UK55.png (742.3KB)
│   │   ├── Selfattention_3K1R_Q495M9_Q9Y6N9.png (284.3KB)
│   │   ├── Selfattention_5BRR_P00750_P05121.png (757.8KB)
│   │   └── Selfattention_6JCK_O14641_O15169.png (223.9KB)
│   ├── mean

In [3]:
import numpy as np
from pathlib import Path

# 1. 标签平衡 + 半区统计（判断 2560 = 1280‖1280 拼接还是单一嵌入）
for split in ["train", "val", "test"]:
    d = np.load(f"data/mean_pooled_scaled/ppi_{split}_mean_pooled_scaled.npz")
    X, y = d["X"], d["y"]
    print(f"{split}: N={len(y)}, 正样本比例={y.mean():.3f}")
    h1, h2 = X[:, :1280], X[:, 1280:]
    print(f"  前半均值/方差: {h1.mean():+.3f}/{h1.std():.3f} | "
          f"后半: {h2.mean():+.3f}/{h2.std():.3f}")
    # 拼接型: 两半统计应高度相似; 且样本内 dim_i 与 dim_{i+1280} 相关性≈0
    corr = np.corrcoef(X[:2000, 5], X[:2000, 1285])[0, 1]
    print(f"  dim5↔dim1285 相关性: {corr:+.3f} (≈0 → 支持拼接假设)")

# 2. 搜寻源文件: 蛋白对列表 + 序列 + 生成脚本
print("\n源文件搜索:")
for pat in ["*.tsv", "*.csv", "*.fasta", "*.fa", "*pairs*", "*gen*", "*extract*", "*embed*"]:
    hits = [p for p in Path(".").rglob(pat)
            if ".git" not in p.parts and "__pycache__" not in p.parts
            and p.suffix != ".pyc"]
    for h in hits[:8]:
        print(f"  [{pat}] {h} ({h.stat().st_size//1024}KB)")


train: N=163192, 正样本比例=0.500
  前半均值/方差: -0.000/1.000 | 后半: -0.000/1.000
  dim5↔dim1285 相关性: -0.019 (≈0 → 支持拼接假设)
val: N=59260, 正样本比例=0.500
  前半均值/方差: -0.003/1.119 | 后半: -0.004/1.113
  dim5↔dim1285 相关性: -0.009 (≈0 → 支持拼接假设)
test: N=52048, 正样本比例=0.500
  前半均值/方差: -0.002/1.122 | 后半: -0.002/1.113
  dim5↔dim1285 相关性: +0.035 (≈0 → 支持拼接假设)

源文件搜索:
  [*.csv] plots\model_modifications.csv (1KB)
  [*.fasta] dataset\human_swissprot_oneliner.fasta (11290KB)
  [*gen*] models\start_agent.py (0KB)
  [*gen*] slurm\start_agent.sh (0KB)
  [*extract*] data\extract16.py (5KB)
  [*extract*] data\extract_esm.py (5KB)
  [*embed*] Embeddings (0KB)
  [*embed*] data\embeddings.py (1KB)
  [*embed*] data\embeddings_1.py (1KB)
  [*embed*] data\output_embeddings (4352KB)
  [*embed*] Embeddings\embeddings_mean.pkl (102801KB)
  [*embed*] Embeddings\embeddings_per_tok.h5 (49068636KB)
  [*embed*] plots\embedding_sizes.pdf (6KB)
  [*embed*] slurm\create_embeddings.sh (0KB)


In [4]:
import h5py
import numpy as np

# 只读元信息, 不加载数据 — 49GB 文件必须用 lazy 模式
f = h5py.File("Embeddings/embeddings_per_tok.h5", "r")

def visit(name, obj):
    if isinstance(obj, h5py.Dataset):
        # 只打印前 5 个 + 抽样几个
        pass

keys = list(f.keys())
print(f"顶层键数量: {len(keys)}")
print(f"前 10 个键: {keys[:10]}")

# 判断组织方式: 顶层就是每蛋白一个数据集? 还是嵌套在组里?
first_key = keys[0]
obj = f[first_key]
if isinstance(obj, h5py.Dataset):
    print(f"\n样本 '{first_key}': shape={obj.shape}, dtype={obj.dtype}")
elif isinstance(obj, h5py.Group):
    sub_keys = list(obj.keys())
    print(f"\n组 '{first_key}' 下有 {len(sub_keys)} 个子对象")
    first_sub = obj[sub_keys[0]]
    if isinstance(first_sub, h5py.Dataset):
        print(f"  样本 '{first_key}/{sub_keys[0]}': shape={first_sub.shape}, dtype={first_sub.dtype}")

# 抽样 20 个键, 检查形状分布
import random
random.seed(42)
sample_keys = random.sample(keys, min(20, len(keys)))
shapes = []
for k in sample_keys:
    d = f[k] if isinstance(f[k], h5py.Dataset) else f[k][list(f[k].keys())[0]]
    shapes.append(d.shape)
Ls = [s[0] for s in shapes]
print(f"\n抽样 20 个: L 范围 {min(Ls)}–{max(Ls)}, dim={set(s[1] for s in shapes)}")
print(f"L 是否全部相同: {len(set(Ls))==1} (若 True, 存在 padding, 需确认长度键/mask)")

# 关键检查: 有没有记录真实长度或 mask?
for candidate in ["lengths", "mask", "seq_lens", "L", "lens", "metadata", "attrs"]:
    if candidate in keys:
        print(f"✓ 发现 '{candidate}' 键")
if not any(c in keys for c in ["lengths","mask","seq_lens","L","lens"]):
    print("⚠ 无长度/mask 键 — padding 未剥离将污染卷积")
f.close()


顶层键数量: 20386
前 10 个键: ['A0A024RBG1', 'A0A075B6H7', 'A0A075B6H8', 'A0A075B6H9', 'A0A075B6I0', 'A0A075B6I1', 'A0A075B6I3', 'A0A075B6I4', 'A0A075B6I6', 'A0A075B6I7']

样本 'A0A024RBG1': shape=(181, 1280), dtype=float32

抽样 20 个: L 范围 120–1022, dim={1280}
L 是否全部相同: False (若 True, 存在 padding, 需确认长度键/mask)
⚠ 无长度/mask 键 — padding 未剥离将污染卷积


In [5]:
import pickle
import numpy as np

with open("Embeddings/embeddings_mean.pkl", "rb") as fp:
    mean_emb = pickle.load(fp)

print(f"类型: {type(mean_emb).__name__}")
if isinstance(mean_emb, dict):
    keys = list(mean_emb.keys())
    print(f"条目数: {len(keys)}")
    print(f"前 10 个键: {keys[:10]}")  # 看键是 Uniprot ID 还是别的
    v = mean_emb[keys[0]]
    print(f"值: {type(v).__name__}, shape={np.asarray(v).shape}, dtype={np.asarray(v).dtype}")
    # 如果键是 Uniprot ID, 这里就是 X 每一行的两半的来源!
elif isinstance(mean_emb, np.ndarray):
    print(f"ndarray shape={mean_emb.shape}, dtype={mean_emb.dtype}")
    print("⚠ 无 ID — 池化嵌入也丢失了对应关系")


类型: dict
条目数: 20386
前 10 个键: ['A0A024RBG1', 'A0A075B6H7', 'A0A075B6H8', 'A0A075B6H9', 'A0A075B6I0', 'A0A075B6I1', 'A0A075B6I3', 'A0A075B6I4', 'A0A075B6I6', 'A0A075B6I7']
值: ndarray, shape=(1280,), dtype=float32


In [6]:
from pathlib import Path

patterns = ["*pair*", "*interaction*", "*positives*", "*negatives*",
            "*labels*", "*ppi*", "*PPI*", "*dataset*", "*split*",
            "*index*", "*mapping*", "*train*", "*test*", "*val*"]
exclude = {".git", "__pycache__", "node_modules", ".ipynb_checkpoints"}

hits = set()
for pat in patterns:
    for p in Path(".").rglob(pat):
        if p.is_file() and not (exclude & set(p.parts)) and p.suffix not in {".pyc", ".so"}:
            hits.add(p)

# 也搜所有 tsv/txt/json/jsonl/pkl 文件
for ext in ["*.tsv", "*.txt", "*.json", "*.jsonl", "*.pkl"]:
    for p in Path(".").rglob(ext):
        if p.is_file() and not (exclude & set(p.parts)):
            hits.add(p)

for p in sorted(hits, key=lambda x: x.stat().st_size, reverse=True)[:30]:
    size = p.stat().st_size / 1024 / 1024
    print(f"{size:8.1f}MB  {p}")


  1594.9MB  data\mean_pooled_scaled\ppi_train_mean_pooled_scaled.npz
   579.2MB  data\mean_pooled_scaled\ppi_val_mean_pooled_scaled.npz
   508.7MB  data\mean_pooled_scaled\ppi_test_mean_pooled_scaled.npz
   250.3MB  data\pca_features_trainfit\ppi_train_pca400_trainfit.npz
   100.4MB  Embeddings\embeddings_mean.pkl
    90.9MB  data\pca_features_trainfit\ppi_val_pca400_trainfit.npz
    79.8MB  data\pca_features_trainfit\ppi_test_pca400_trainfit.npz
    26.1MB  data\pca_features_trainfit\ppi_train_pca40_trainfit.npz
     9.5MB  data\pca_features_trainfit\ppi_val_pca40_trainfit.npz
     8.3MB  data\pca_features_trainfit\ppi_test_pca40_trainfit.npz
     1.1MB  dataset\Intra1_pos_rr.txt
     1.1MB  dataset\Intra1_neg_rr.txt
     0.4MB  dataset\Intra0_neg_rr.txt
     0.4MB  dataset\Intra0_pos_rr.txt
     0.3MB  dataset\Intra2_neg_rr.txt
     0.3MB  dataset\Intra2_pos_rr.txt
     0.2MB  plots\contact_map_test.png
     0.0MB  test_model.py
     0.0MB  test.py
     0.0MB  plots\visualize_dataset

In [7]:
for fp in ["data/extract16.py", "data/extract_esm.py",
           "data/embeddings.py", "data/embeddings_1.py"]:
    print(f"\n{'='*70}\n{fp}\n{'='*70}")
    with open(fp, encoding="utf-8", errors="replace") as f:
        print(f.read())



data/extract16.py
#!/usr/bin/env python3 -u
# Copyright (c) Meta Platforms, Inc. and affiliates.
#
# This source code is licensed under the MIT license found in the
# LICENSE file in the root directory of this source tree.

import argparse
import pathlib

import torch

from esm import Alphabet, FastaBatchedDataset, ProteinBertModel, pretrained, MSATransformer


def create_parser():
    parser = argparse.ArgumentParser(
        description="Extract per-token representations and model outputs for sequences in a FASTA file"  # noqa
    )

    parser.add_argument(
        "model_location",
        type=str,
        help="PyTorch model file OR name of pretrained model to download (see README for models)",
    )
    parser.add_argument(
        "fasta_file",
        type=pathlib.Path,
        help="FASTA file on which to extract representations",
    )
    parser.add_argument(
        "output_dir",
        type=pathlib.Path,
        help="output directory for extracted representations",
   

In [3]:
import subprocess
import torch, psutil
from pathlib import Path

print(f"GPU: {torch.cuda.get_device_name(0)}, "
      f"显存 {torch.cuda.get_device_properties(0).total_memory/1024**3:.0f}GB, "
      f"算力 {torch.cuda.get_device_capability(0)}")
print(f"RAM 总量 {psutil.virtual_memory().total/1024**3:.0f}GB, "
      f"可用 {psutil.virtual_memory().available/1024**3:.0f}GB")

h5 = Path(r"D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)"
          r"\Embeddings\embeddings_per_tok.h5")
print(f"h5 大小: {h5.stat().st_size/1024**3:.1f}GB, 所在盘: {str(h5)[:3]}")

# 磁盘类型 (SSD / HDD) —— 通过 PowerShell 查询
try:
    out = subprocess.run(
        ["powershell", "-NoProfile", "-Command",
         "Get-PhysicalDisk | Select-Object DeviceId, MediaType, BusType | "
         "Format-Table -AutoSize"],
        capture_output=True, text=True, timeout=15)
    print("磁盘类型:\n" + out.stdout)
except Exception as e:
    print(f"自动查询失败({e})——请在任务管理器→性能→对应磁盘的右上角查看"
          f"是否标注 'SSD' 或 '硬盘' ")


GPU: NVIDIA GeForce RTX 4060 Laptop GPU, 显存 8GB, 算力 (8, 9)
RAM 总量 32GB, 可用 15GB
h5 大小: 46.8GB, 所在盘: D:\
磁盘类型:

DeviceId MediaType BusType
-------- --------- -------
0        SSD       NVMe   





In [5]:
import hashlib, datetime
from check import CKPT
st = CKPT.stat()
print(CKPT.name, datetime.datetime.fromtimestamp(st.st_mtime),
      hashlib.md5(CKPT.read_bytes()).hexdigest()[:8])


2d_selfattention_best.pt 2026-09-07 18:55:21.672228 1cedd762


In [7]:
# -*- coding: utf-8 -*-
"""collect_cache.py —— 加载已验证 checkpoint，per-sample 收集 val/test 分数并缓存
   运行一次（约40分钟），之后 calibrate_v2.py 与 model_healthcheck.py 全部秒级可用"""
import hashlib
import numpy as np
import torch
from pathlib import Path

from check import (CKPT, BASE_DIR, EMBED_DIM, NUM_HEADS, H3, FF_DIM, DROPOUT,
                   POOLING, KERNEL_SIZE, DEVICE,
                   VAL_POS, VAL_NEG, TEST_POS, TEST_NEG,
                   SelfAttInteraction, load_split, Split, collect_scores)

# ---- 身份确认（必须仍为 1cedd762）----
md5 = hashlib.md5(CKPT.read_bytes()).hexdigest()[:8]
assert md5 == "1cedd762", f"checkpoint 已变化: {md5}（先弄清是哪次训练的再继续）"
print(f"checkpoint 身份确认: md5={md5}\n")

model = SelfAttInteraction(EMBED_DIM, NUM_HEADS, h3=H3, dropout=DROPOUT,
                           ff_dim=FF_DIM, pooling=POOLING,
                           kernel_size=KERNEL_SIZE).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))  # weights_only 消除 FutureWarning
print("加载成功，开始收集分数（per-sample，论文协议）…")

pv, yv = collect_scores(model, Split(load_split(VAL_POS, VAL_NEG)))
print(f"val 完成: aupr={average_precision_score(yv, pv):.3f}")
pt, yt = collect_scores(model, Split(load_split(TEST_POS, TEST_NEG)))
from sklearn.metrics import average_precision_score
print(f"test 完成: aupr={average_precision_score(yt, pt):.3f}")

np.savez(BASE_DIR / "score_cache.npz", pv=pv, yv=yv, pt=pt, yt=yt)
print(f"\n缓存已写入: {BASE_DIR / 'score_cache.npz'}")


checkpoint 身份确认: md5=1cedd762

加载成功，开始收集分数（per-sample，论文协议）…
[Intra0_pos_rr.txt] 解析 29630 行
[Intra0_neg_rr.txt] 解析 29630 行
打开嵌入文件: D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)\Embeddings\embeddings_per_tok.h5
h5 中共 20386 个蛋白
[Intra0_pos_rr+Intra0_neg_rr] 样本=46421 (正=23412, 负=23009)，缺失蛋白 0 个已剔除


d:\pythonprojects\practice-github\torch_final\Lib\site-packages\torch\nn\modules\conv.py:549: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv2d(


val 完成: aupr=0.605
[Intra2_pos_rr.txt] 解析 26024 行
[Intra2_neg_rr.txt] 解析 26024 行
[Intra2_pos_rr+Intra2_neg_rr] 样本=41100 (正=20642, 负=20458)，缺失蛋白 0 个已剔除
test 完成: aupr=0.625

缓存已写入: D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)\score_cache.npz


In [9]:
# -*- coding: utf-8 -*-
"""
final_report.py —— 终版报告生成器（单次运行）
流程: md5校验 → 加载 → per-sample收集 val/test(存score_cache.npz)
      → 三准则阈值校准(含退化守卫) → 分数健康终检 → 论文对照表
之后再跑任何分析: 直接 np.load(score_cache.npz), 秒级
"""
import re, random, hashlib, datetime
from pathlib import Path
import h5py, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.spectral_norm as spectral_norm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score,
                             matthews_corrcoef, confusion_matrix, roc_auc_score)

# ================= 配置（与 check.py 一致） =================
BASE_DIR = Path(r"D:/pythonprojects/practice-github/PPI_prediction(gold-standard dataset)")
EMB_H5   = BASE_DIR / "Embeddings" / "embeddings_per_tok.h5"
CKPT     = BASE_DIR / "checkpoints" / "2d_selfattention_best.pt"
CACHE    = BASE_DIR / "score_cache.npz"
VAL_POS,  VAL_NEG  = BASE_DIR/"dataset"/"Intra0_pos_rr.txt", BASE_DIR/"dataset"/"Intra0_neg_rr.txt"
TEST_POS, TEST_NEG = BASE_DIR/"dataset"/"Intra2_pos_rr.txt", BASE_DIR/"dataset"/"Intra2_neg_rr.txt"
MD5_EXPECT = "1cedd762"
EMBED_DIM, NUM_HEADS, H3, FF_DIM, DROPOUT = 1280, 8, 64, 256, 0.2
POOLING, KERNEL_SIZE, MAX_LEN = 'max', 2, 1000
BATCH_SIZE, SEED = 16, 42
PAPER = {"acc":0.616,"precision":0.611,"recall":0.553,"f1":0.591,"aupr":0.641}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ================= 嵌入读取（per-sample，无缓存） =================
_H5, _KEYSET, _KEYMAP, _LEN = None, None, {}, {}
def open_h5():
    global _H5, _KEYSET
    if _H5 is None:
        _H5 = h5py.File(EMB_H5, "r")
        keys = [k.decode() if isinstance(k, bytes) else k for k in _H5.keys()]
        _KEYSET = set(keys)
        for k in keys:
            for c in (k.split("-")[0], k.split("|")[0].split("/")[-1],
                      k.split("_")[0] if "_" in k else None):
                if c and c not in _KEYSET and c not in _KEYMAP: _KEYMAP[c] = k
    return _H5
def resolve_key(n):
    if n in _KEYSET: return n
    if n in _KEYMAP: return _KEYMAP[n]
    for c in (n.split("-")[0], n.split("|")[0].split("/")[-1],
              n.split("_")[0] if "_" in n else None):
        if c and c in _KEYSET: return c
        if c and c in _KEYMAP: return _KEYMAP[c]
    return None
def protein_len(n):
    if n in _LEN: return _LEN[n]
    k = resolve_key(n)
    if k is None: _LEN[n] = None; return None
    o = open_h5()[k]
    if isinstance(o, h5py.Group): o = o[list(o.keys())[0]]
    _LEN[n] = int(o.shape[-2]); return _LEN[n]
def get_emb(n):
    o = open_h5()[resolve_key(n)]
    if isinstance(o, h5py.Group): o = o[list(o.keys())[-1]]
    t = torch.from_numpy(np.array(o[()], dtype=np.float32))
    return t.squeeze(0) if (t.ndim == 3 and t.shape[0] == 1) else t

# ================= 数据加载（已验证版） =================
_ID_RE = re.compile(r"^[OPQ][0-9][A-Z0-9]{3}[0-9](-\d+)?$"
                    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}(-\d+)?$")
_LABEL_KW  = {"interaction","label","target","y","class","is_interaction"}
_HEADER_KW = _LABEL_KW | {"name1","name2","protein1","protein2","id1","id2",
                          "uniprot1","uniprot2","protein_a","protein_b"}
_BINARY = {"0","1","0.0","1.0","true","false","pos","neg","positive","negative"}
def _lab(v, d):
    s = str(v).strip().lower()
    return 1.0 if s in ("1","true","pos","positive") else 0.0 if s in ("0","false","neg","negative") else d
def _bincol(df):
    bc, bf = None, 0.0
    for j in range(df.shape[1]):
        f = df.iloc[:,j].astype(str).str.strip().str.lower().isin(_BINARY).mean()
        if f > bf: bc, bf = j, f
    return bc if bf >= 0.8 else None
def _read(path, dlab):
    df = None
    for sep in (None,"\t",",",r"\s+",";"):
        try:
            d = pd.read_csv(path, sep=sep, engine="python", header=None,
                            dtype=str, keep_default_na=False)
        except Exception: continue
        if d.shape[1] >= 2: df = d; break
    df = df[(df != "").any(axis=1)].reset_index(drop=True)
    r0 = [str(v).strip().lower() for v in df.iloc[0]]
    lc = next((j for j,v in enumerate(r0) if v in _LABEL_KW), None)
    if lc is not None or sum(v in _HEADER_KW for v in r0) >= 2:
        if lc is None: lc = _bincol(df.iloc[1:])
        first = sum(_ID_RE.match(v.upper()) for j,v in enumerate(r0) if j != lc) > 0
        data = df if first else df.iloc[1:]
    else:
        data, lc = df, (_bincol(df) if df.shape[1] >= 3 else None)
    ncs = [j for j in range(df.shape[1]) if j != lc][:2] if lc is not None else [0,1]
    out = pd.DataFrame({"name1": data.iloc[:,ncs[0]].astype(str).str.strip(),
                        "name2": data.iloc[:,ncs[1]].astype(str).str.strip()})
    out["interaction"] = (data.iloc[:,lc].apply(lambda v: _lab(v, dlab))
                          if lc is not None else float(dlab))
    return out[(out["name1"]!="")&(out["name2"]!="")].reset_index(drop=True)
def load_split(pos, neg):
    df = pd.concat([_read(pos,1.0), _read(neg,0.0)], ignore_index=True)
    df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    open_h5(); keep, miss = [], set()
    for i,(a,b) in enumerate(zip(df["name1"], df["name2"])):
        la, lb = protein_len(a), protein_len(b)
        if la is None: miss.add(a); continue
        if lb is None: miss.add(b); continue
        if la <= MAX_LEN and lb <= MAX_LEN: keep.append(i)
    df = df.iloc[keep].reset_index(drop=True)
    print(f"[{pos.stem}+{neg.stem}] 样本={len(df)} 正={int(df['interaction'].sum())} "
          f"缺失剔除={len(miss)}")
    return df
class Split:
    def __init__(s, df):
        s.n1, s.n2 = df["name1"].astype(str).tolist(), df["name2"].astype(str).tolist()
        s.y = df["interaction"].astype(float).tolist()
    def __len__(s): return len(s.y)
    def __getitem__(s, idx):
        return {"name1":[s.n1[i] for i in idx], "name2":[s.n2[i] for i in idx],
                "interaction":[s.y[i] for i in idx]}

# ================= 模型（与 checkpoint 严格同构） =================
class Attention(nn.Module):
    def __init__(s, hid, nh, do):
        super().__init__(); s.hid, s.nh = hid, nh
        s.w_q = spectral_norm(nn.Linear(hid,hid)); s.w_k = spectral_norm(nn.Linear(hid,hid))
        s.w_v = spectral_norm(nn.Linear(hid,hid)); s.fc  = spectral_norm(nn.Linear(hid,hid))
        s.do = nn.Dropout(do); s.scale = torch.sqrt(torch.FloatTensor([hid//nh])).to(DEVICE)
    def forward(s, q, k, v, mask=None):
        b = q.shape[0]
        Q = s.w_q(q).view(b,-1,s.nh,s.hid//s.nh).permute(0,2,1,3)
        K = s.w_k(k).view(b,-1,s.nh,s.hid//s.nh).permute(0,2,1,3)
        V = s.w_v(v).view(b,-1,s.nh,s.hid//s.nh).permute(0,2,1,3)
        e = torch.matmul(Q, K.permute(0,1,3,2)) / s.scale
        if mask is not None: e = e.masked_fill(mask==0, -1e10)
        x = torch.matmul(s.do(F.softmax(e, dim=-1)), V)
        x = x.permute(0,2,1,3).contiguous().view(b,-1,s.hid)
        return s.fc(x)
class FF(nn.Module):
    def __init__(s, hid, ff, do):
        super().__init__()
        s.fc_1 = spectral_norm(nn.Linear(hid, ff))   # ★ 键名对齐: fc_1
        s.fc_2 = spectral_norm(nn.Linear(ff, hid))   # ★ 键名对齐: fc_2
        s.do = nn.Dropout(do); s.a = nn.SiLU()
    def forward(s, x):
        return s.fc_2(s.do(s.a(s.fc_1(x))))
class Encoder(nn.Module):
    def __init__(s, hid, nh, ff, do):
        super().__init__()
        s.ln1, s.ln2 = nn.LayerNorm(hid), nn.LayerNorm(hid)
        s.d1, s.d2 = nn.Dropout(do), nn.Dropout(do)
        s.sa, s.ff = Attention(hid,nh,do), FF(hid,ff,do)
    def forward(s, x, mask=None):
        x = s.ln1(x + s.d1(s.sa(x,x,x,mask)))
        return s.ln2(x + s.d2(s.ff(x)))
class SelfAtt(nn.Module):
    def __init__(s, D, nh, h3=64, do=0.2, ff=256, pooling='max', ks=2):
        super().__init__()
        h, h2 = D//4, D//16
        s.encoder   = Encoder(h3, nh, ff, do)
        s.multihead = Attention(h3, nh, do)          # ★ 训练脚本有此层（forward 未用），仅为对齐 state_dict
        s.conv      = nn.Conv2d(h3, 1, kernel_size=ks, padding='same')
        s.map_norm  = nn.InstanceNorm2d(1, affine=True)   # ★ 键名: map_norm
        s.pool      = nn.MaxPool2d(ks) if pooling == 'max' else nn.AvgPool2d(ks)
        s.fc1 = nn.Linear(D, h); s.fc2 = nn.Linear(h, h2); s.fc3 = nn.Linear(h2, h3)  # ★ fc1/2/3
    def forward(s, p1, p2, m1=None, m2=None):
        x1 = p1.float().unsqueeze(0); x2 = p2.float().unsqueeze(0)
        x1 = F.relu(s.fc3(F.relu(s.fc2(F.relu(s.fc1(x1))))))
        x2 = F.relu(s.fc3(F.relu(s.fc2(F.relu(s.fc1(x2))))))
        x1 = s.encoder(x1, m1); x2 = s.encoder(x2, m2)
        mat = torch.einsum('bik,bjk->bijk', x1, x2).permute(0, 3, 1, 2)
        mat = s.map_norm(s.conv(mat))
        x = s.pool(mat).flatten()
        k = max(1, int(0.01 * x.numel()))
        return torch.sigmoid(torch.topk(x, k).values.mean())[None], mat

# ================= 收集（per-sample + model.eval，含 anchor 复核） =================
def batch_iter(model, batch):
    ps = []
    for i in range(len(batch["interaction"])):
        s1 = get_emb(batch["name1"][i]).to(DEVICE)
        s2 = get_emb(batch["name2"][i]).to(DEVICE)
        p, _ = model(s1, s2); ps.append(p)
    return torch.stack(ps).view(-1)
@torch.no_grad()
def collect(model, ds):
    model.eval()                                   # ★ 每次评估前显式调用
    P, Y = [], []
    for s in range(0, len(ds), BATCH_SIZE):
        b = ds[list(range(s, min(s+BATCH_SIZE, len(ds))))]
        P.append(batch_iter(model, b).cpu()); Y += b["interaction"]
    return torch.cat(P).numpy(), np.asarray(Y)

# ================= 主流程 =================
def main():
    md5 = hashlib.md5(CKPT.read_bytes()).hexdigest()[:8]
    assert md5 == MD5_EXPECT, f"checkpoint 已变: {md5}"
    print(f"md5 校验通过: {md5} @ {datetime.datetime.fromtimestamp(CKPT.stat().st_mtime)}\n")

    model = SelfAtt(EMBED_DIM, NUM_HEADS, H3, DROPOUT, FF_DIM, POOLING, KERNEL_SIZE).to(DEVICE)
    model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))

    pv, yv = collect(model, Split(load_split(VAL_POS, VAL_NEG)))
    av = average_precision_score(yv, pv)
    print(f"[anchor] val aupr={av:.3f}（预期 0.605，不符→路径有差异，先停）")
    pt, yt = collect(model, Split(load_split(TEST_POS, TEST_NEG)))
    at = average_precision_score(yt, pt)
    print(f"[anchor] test aupr={at:.3f}（预期 0.625）\n")
    np.savez(CACHE, pv=pv, yv=yv, pt=pt, yt=yt)
    print(f"缓存已存: {CACHE}\n")

    # ---- 三准则校准（只在 val 上选）+ 退化守卫 ----
    ts = np.round(np.arange(0.05, 0.96, 0.01), 2)
    youden = lambda y, pb: (lambda c: c[3]/max(1,c[3]+c[2]) - c[1]/max(1,c[1]+c[0]))(
        confusion_matrix(y, pb, labels=[0,1]).ravel())
    mcc_v = [matthews_corrcoef(yv, pv>=t) for t in ts]
    jd_v  = [youden(yv, pv>=t) for t in ts]
    cands = {"@MCC*": float(ts[int(np.argmax(mcc_v))]),
             "@Youden*": float(ts[int(np.argmax(jd_v))]),
             "@中位数": float(np.quantile(pv, 0.5))}
    print("[候选阈值]")
    for n,t in cands.items():
        g = "⚠全正退化" if (pv>=t).mean() > 0.9 else "ok"
        print(f"  {n:>9}: t={t:.2f} val预测正率={(pv>=t).mean():.2f} [{g}]")

    # ---- 健康终检 ----
    print(f"\n[健康终检] AUROC={roc_auc_score(yt, pt):.3f} | "
          f"分离度 Δmean={pt[yt==1].mean()-pt[yt==0].mean():+.3f}")
    for k in (1000, 5000, 10000):
        print(f"  P@{k:>5} = {yt[np.argsort(-pt)[:k]].mean():.3f}（基线0.500）")
    qs = np.quantile(pt, np.linspace(0,1,11))
    rates = [yt[(pt>=qs[i])&(pt<=qs[i+1])].mean() for i in range(10)]
    mono = all(rates[i] <= rates[i+1] + 0.02 for i in range(9))
    print("  十分位正率单调性:", "通过 ✓" if mono else "未通过 ✗")

    # ---- 论文对照表 ----
    row = lambda n, t: (lambda m: print(f"{n:<14}{t:>6.2f}{m['acc']:>8.3f}"
        f"{m['precision']:>8.3f}{m['recall']:>8.3f}{m['f1']:>8.3f}"
        f"{m['mcc']:>8.3f}"))({"acc":accuracy_score(yt,pt>=t),
        "precision":precision_score(yt,pt>=t,zero_division=0),
        "recall":recall_score(yt,pt>=t,zero_division=0),
        "f1":f1_score(yt,pt>=t,zero_division=0),
        "mcc":matthews_corrcoef(yt,pt>=t)} if t is not None else None) \
        if t is not None else print(f"{n:<14}{'—':>6}{'—':>8}{'—':>8}{'—':>8}{'—':>8}{'—':>8}")
    print(f"\n aupr: test={at:.3f} vs 论文={PAPER['aupr']:.3f} ({at-PAPER['aupr']:+.3f})\n")
    print(f"{'口径':<14}{'t':>6}{'acc':>8}{'prec':>8}{'rec':>8}{'f1':>8}{'mcc':>8}")
    print(f"{'论文@0.5':<14}{'':>6}{PAPER['acc']:>8.3f}{PAPER['precision']:>8.3f}"
          f"{PAPER['recall']:>8.3f}{PAPER['f1']:>8.3f}{'—':>8}")
    row("@0.5 保真", 0.50)
    for n,t in cands.items(): row(n, t)

if __name__ == "__main__":
    main()


md5 校验通过: 1cedd762 @ 2026-09-07 18:55:21.672228

[Intra0_pos_rr+Intra0_neg_rr] 样本=46421 正=23412 缺失剔除=0
[anchor] val aupr=0.605（预期 0.605，不符→路径有差异，先停）
[Intra2_pos_rr+Intra2_neg_rr] 样本=41100 正=20642 缺失剔除=0
[anchor] test aupr=0.625（预期 0.625）

缓存已存: D:\pythonprojects\practice-github\PPI_prediction(gold-standard dataset)\score_cache.npz

[候选阈值]
      @MCC*: t=0.56 val预测正率=0.40 [ok]
   @Youden*: t=0.54 val预测正率=0.43 [ok]
       @中位数: t=0.50 val预测正率=0.50 [ok]

[健康终检] AUROC=0.614 | 分离度 Δmean=+0.083
  P@ 1000 = 0.823（基线0.500）
  P@ 5000 = 0.733（基线0.500）
  P@10000 = 0.659（基线0.500）
  十分位正率单调性: 通过 ✓

 aupr: test=0.625 vs 论文=0.641 (-0.016)

口径                 t     acc    prec     rec      f1     mcc
论文@0.5                 0.616   0.611   0.553   0.591       —
@0.5 保真         0.50   0.578   0.595   0.500   0.543   0.159
@MCC*           0.56   0.578   0.621   0.411   0.495   0.167
@Youden*        0.54   0.578   0.612   0.438   0.511   0.165
@中位数            0.50   0.579   0.597   0.496   0.542   0.161


In [10]:
# -*- coding: utf-8 -*-
"""
multiseed_train.py —— 种子扫描 + 方差聚合（复现收尾）
用法:
  python multiseed_train.py 42 7 2024      # 依次训练三个种子（各约1.5-2.5h）
  python multiseed_train.py 42             # 只跑一个种子
  python multiseed_train.py agg            # 只聚合已有结果（秒级）
配置（环境变量，默认=产出0.625那轮的已知值）:
  PP_LR=1e-4  PP_FRAC=1.0  PP_BATCH=16  PP_EPOCHS=60  PP_PATIENCE=8
  ★ 运行前核对: 打开你产出0.625那轮的训练脚本, 确认 LR / BATCH_SIZE / SUBSET_FRAC
    的实际值并设为相同 env —— 种子扫描只允许改 PP_SEED 这一个变量
断点续跑: results_seed_sweep.jsonl 中已有的种子自动跳过
"""
import os, re, sys, gc, json, random, hashlib, datetime
from pathlib import Path
import h5py, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.spectral_norm as spectral_norm
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, average_precision_score, matthews_corrcoef)

# ================= 配置 =================
BASE_DIR = Path(r"D:/pythonprojects/practice-github/PPI_prediction(gold-standard dataset)")
EMB_H5   = BASE_DIR / "Embeddings" / "embeddings_per_tok.h5"
CKPT_DIR = BASE_DIR / "checkpoints"
RESULTS  = BASE_DIR / "results_seed_sweep.jsonl"
TRAIN_POS, TRAIN_NEG = BASE_DIR/"dataset"/"Intra1_pos_rr.txt", BASE_DIR/"dataset"/"Intra1_neg_rr.txt"
VAL_POS,   VAL_NEG   = BASE_DIR/"dataset"/"Intra0_pos_rr.txt", BASE_DIR/"dataset"/"Intra0_neg_rr.txt"
TEST_POS,  TEST_NEG  = BASE_DIR/"dataset"/"Intra2_pos_rr.txt", BASE_DIR/"dataset"/"Intra2_neg_rr.txt"
MD5_ANCHOR = "1cedd762"      # 原始 checkpoint, 本脚本不触碰

LR       = float(os.environ.get("PP_LR", "1e-4"))
FRAC     = float(os.environ.get("PP_FRAC", "1.0"))
BATCH    = int(os.environ.get("PP_BATCH", "16"))
EPOCHS   = int(os.environ.get("PP_EPOCHS", "60"))
PATIENCE = int(os.environ.get("PP_PATIENCE", "8"))

EMBED_DIM, NUM_HEADS, H3, FF_DIM, DROPOUT = 1280, 8, 64, 256, 0.2
POOLING, KERNEL_SIZE, MAX_LEN = 'max', 2, 1000
PAPER = {"aupr":0.641,"acc":0.616,"f1":0.591}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"协议: LR={LR} FRAC={FRAC} BATCH={BATCH} PATIENCE={PATIENCE} "
      f"选点=val_mcc | Device={DEVICE}\n")

# ================= 嵌入（已验证版） =================
_H5, _KEYSET, _KEYMAP, _LEN = None, None, {}, {}
def open_h5():
    global _H5, _KEYSET
    if _H5 is None:
        _H5 = h5py.File(EMB_H5, "r")
        keys = [k.decode() if isinstance(k, bytes) else k for k in _H5.keys()]
        _KEYSET = set(keys)
        for k in keys:
            for c in (k.split("-")[0], k.split("|")[0].split("/")[-1],
                      k.split("_")[0] if "_" in k else None):
                if c and c not in _KEYSET and c not in _KEYMAP: _KEYMAP[c] = k
    return _H5
def resolve_key(n):
    if n in _KEYSET: return n
    if n in _KEYMAP: return _KEYMAP[n]
    for c in (n.split("-")[0], n.split("|")[0].split("/")[-1],
              n.split("_")[0] if "_" in n else None):
        if c and c in _KEYSET: return c
        if c and c in _KEYMAP: return _KEYMAP[c]
    return None
def protein_len(n):
    if n in _LEN: return _LEN[n]
    k = resolve_key(n)
    if k is None: _LEN[n] = None; return None
    o = open_h5()[k]
    if isinstance(o, h5py.Group): o = o[list(o.keys())[0]]
    _LEN[n] = int(o.shape[-2]); return _LEN[n]
def get_emb(n):
    o = open_h5()[resolve_key(n)]
    if isinstance(o, h5py.Group): o = o[list(o.keys())[-1]]
    t = torch.from_numpy(np.array(o[()], dtype=np.float32))
    return t.squeeze(0) if (t.ndim == 3 and t.shape[0] == 1) else t

# ================= 数据（已验证版） =================
_ID_RE = re.compile(r"^[OPQ][0-9][A-Z0-9]{3}[0-9](-\d+)?$"
                    r"|^[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}(-\d+)?$")
_LABEL_KW  = {"interaction","label","target","y","class","is_interaction"}
_HEADER_KW = _LABEL_KW | {"name1","name2","protein1","protein2","id1","id2",
                          "uniprot1","uniprot2","protein_a","protein_b"}
_BINARY = {"0","1","0.0","1.0","true","false","pos","neg","positive","negative"}
def _lab(v, d):
    s = str(v).strip().lower()
    return 1.0 if s in ("1","true","pos","positive") else \
           0.0 if s in ("0","false","neg","negative") else d
def _bincol(df):
    bc, bf = None, 0.0
    for j in range(df.shape[1]):
        f = df.iloc[:,j].astype(str).str.strip().str.lower().isin(_BINARY).mean()
        if f > bf: bc, bf = j, f
    return bc if bf >= 0.8 else None
def _read(path, dlab):
    df = None
    for sep in (None,"\t",",",r"\s+",";"):
        try:
            d = pd.read_csv(path, sep=sep, engine="python", header=None,
                            dtype=str, keep_default_na=False)
        except Exception: continue
        if d.shape[1] >= 2: df = d; break
    df = df[(df != "").any(axis=1)].reset_index(drop=True)
    r0 = [str(v).strip().lower() for v in df.iloc[0]]
    lc = next((j for j,v in enumerate(r0) if v in _LABEL_KW), None)
    if lc is not None or sum(v in _HEADER_KW for v in r0) >= 2:
        if lc is None: lc = _bincol(df.iloc[1:])
        first = sum(_ID_RE.match(v.upper()) for j,v in enumerate(r0) if j != lc) > 0
        data = df if first else df.iloc[1:]
    else:
        data, lc = df, (_bincol(df) if df.shape[1] >= 3 else None)
    ncs = [j for j in range(df.shape[1]) if j != lc][:2] if lc is not None else [0,1]
    out = pd.DataFrame({"name1": data.iloc[:,ncs[0]].astype(str).str.strip(),
                        "name2": data.iloc[:,ncs[1]].astype(str).str.strip()})
    out["interaction"] = (data.iloc[:,lc].apply(lambda v: _lab(v, dlab))
                          if lc is not None else float(dlab))
    return out[(out["name1"]!="")&(out["name2"]!="")].reset_index(drop=True)
def load_split(pos, neg):
    df = pd.concat([_read(pos,1.0), _read(neg,0.0)], ignore_index=True)
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    open_h5(); keep, miss = [], set()
    for i,(a,b) in enumerate(zip(df["name1"], df["name2"])):
        la, lb = protein_len(a), protein_len(b)
        if la is None: miss.add(a); continue
        if lb is None: miss.add(b); continue
        if la <= MAX_LEN and lb <= MAX_LEN: keep.append(i)
    df = df.iloc[keep].reset_index(drop=True)
    print(f"[{pos.stem}+{neg.stem}] n={len(df)} 正={int(df['interaction'].sum())}")
    return df
class Split:
    def __init__(s, df):
        s.n1, s.n2 = df["name1"].astype(str).tolist(), df["name2"].astype(str).tolist()
        s.y = df["interaction"].astype(float).tolist()
    def __len__(s): return len(s.y)
    def __getitem__(s, idx):
        return {"name1":[s.n1[i] for i in idx], "name2":[s.n2[i] for i in idx],
                "interaction":[s.y[i] for i in idx]}

# ================= 模型（键名对齐 1cedd762） =================
class Attention(nn.Module):
    def __init__(s, hid, nh, do):
        super().__init__(); s.hid, s.nh = hid, nh
        s.w_q = spectral_norm(nn.Linear(hid,hid)); s.w_k = spectral_norm(nn.Linear(hid,hid))
        s.w_v = spectral_norm(nn.Linear(hid,hid)); s.fc  = spectral_norm(nn.Linear(hid,hid))
        s.do = nn.Dropout(do); s.scale = torch.sqrt(torch.FloatTensor([hid//nh])).to(DEVICE)
    def forward(s, q, k, v, mask=None):
        b = q.shape[0]
        Q = s.w_q(q).view(b,-1,s.nh,s.hid//s.nh).permute(0,2,1,3)
        K = s.w_k(k).view(b,-1,s.nh,s.hid//s.nh).permute(0,2,1,3)
        V = s.w_v(v).view(b,-1,s.nh,s.hid//s.nh).permute(0,2,1,3)
        e = torch.matmul(Q, K.permute(0,1,3,2)) / s.scale
        if mask is not None: e = e.masked_fill(mask==0, -1e10)
        x = torch.matmul(s.do(F.softmax(e, dim=-1)), V)
        x = x.permute(0,2,1,3).contiguous().view(b,-1,s.hid)
        return s.fc(x)
class FF(nn.Module):
    def __init__(s, hid, ff, do):
        super().__init__()
        s.fc_1 = spectral_norm(nn.Linear(hid,ff)); s.fc_2 = spectral_norm(nn.Linear(ff,hid))
        s.do = nn.Dropout(do); s.a = nn.SiLU()
    def forward(s, x): return s.fc_2(s.do(s.a(s.fc_1(x))))
class Encoder(nn.Module):
    def __init__(s, hid, nh, ff, do):
        super().__init__()
        s.ln1, s.ln2 = nn.LayerNorm(hid), nn.LayerNorm(hid)
        s.d1, s.d2 = nn.Dropout(do), nn.Dropout(do)
        s.sa, s.ff = Attention(hid,nh,do), FF(hid,ff,do)
    def forward(s, x, mask=None):
        x = s.ln1(x + s.d1(s.sa(x,x,x,mask)))
        return s.ln2(x + s.d2(s.ff(x)))
class SelfAtt(nn.Module):
    def __init__(s, D, nh, h3=64, do=0.2, ff=256, pooling='max', ks=2):
        super().__init__()
        h, h2 = D//4, D//16
        s.encoder   = Encoder(h3,nh,ff,do)
        s.multihead = Attention(h3,nh,do)      # state_dict 对齐项（forward 未用）
        s.conv      = nn.Conv2d(h3,1,kernel_size=ks,padding='same')
        s.map_norm  = nn.InstanceNorm2d(1, affine=True)
        s.pool      = nn.MaxPool2d(ks) if pooling=='max' else nn.AvgPool2d(ks)
        s.fc1, s.fc2, s.fc3 = nn.Linear(D,h), nn.Linear(h,h2), nn.Linear(h2,h3)
    def forward(s, p1, p2, m1=None, m2=None):
        x1 = p1.float().unsqueeze(0); x2 = p2.float().unsqueeze(0)
        x1 = F.relu(s.fc3(F.relu(s.fc2(F.relu(s.fc1(x1))))))
        x2 = F.relu(s.fc3(F.relu(s.fc2(F.relu(s.fc1(x2))))))
        x1 = s.encoder(x1, m1); x2 = s.encoder(x2, m2)
        mat = torch.einsum('bik,bjk->bijk', x1, x2).permute(0,3,1,2)
        mat = s.map_norm(s.conv(mat))
        x = s.pool(mat).flatten()
        k = max(1, int(0.01 * x.numel()))
        return torch.sigmoid(torch.topk(x, k).values.mean())[None], mat

# ================= 评估（per-sample + eval，已验证） =================
def batch_iter(model, batch):
    ps = []
    for i in range(len(batch["interaction"])):
        p, _ = model(get_emb(batch["name1"][i]).to(DEVICE),
                     get_emb(batch["name2"][i]).to(DEVICE))
        ps.append(p)
    return torch.stack(ps).view(-1)
@torch.no_grad()
def collect(model, ds):
    model.eval()                               # ★ 每次评估显式调用
    P, Y = [], []
    for s in range(0, len(ds), BATCH):
        b = ds[list(range(s, min(s+BATCH, len(ds))))]
        P.append(batch_iter(model, b).cpu()); Y += b["interaction"]
    return torch.cat(P).numpy(), np.asarray(Y)
def metrics(y, p, t=0.5):
    pb = p >= t
    return {"acc":accuracy_score(y,pb), "precision":precision_score(y,pb,zero_division=0),
            "recall":recall_score(y,pb,zero_division=0), "f1":f1_score(y,pb,zero_division=0),
            "mcc":matthews_corrcoef(y,pb), "aupr":average_precision_score(y,p)}

def stratified_idx(ds, frac):
    pos = [i for i,v in enumerate(ds.y) if v==1.0]
    neg = [i for i,v in enumerate(ds.y) if v==0.0]
    idx = (random.sample(pos, max(1,int(frac*len(pos))))
           + random.sample(neg, max(1,int(frac*len(neg)))))
    random.shuffle(idx); return idx

# ================= 单种子训练 =================
def train_one_seed(seed, train_ds, val_ds, test_ds):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"\n{'='*60}\nSEED {seed} | {datetime.datetime.now():%H:%M:%S}\n{'='*60}")

    model = SelfAtt(EMBED_DIM, NUM_HEADS, H3, DROPOUT, FF_DIM, POOLING, KERNEL_SIZE).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.BCELoss()
    ckpt = CKPT_DIR / f"2d_selfattention_seed{seed}.pt"

    best_mcc, wait, best_ep = -1.0, 0, 0
    t0 = datetime.datetime.now()
    for ep in range(1, EPOCHS+1):
        model.train()
        idxs = stratified_idx(train_ds, FRAC)
        ep_loss, nb = 0.0, 0
        for s in range(0, len(idxs), BATCH):
            b = train_ds[idxs[s:s+BATCH]]
            preds = batch_iter(model, b)
            labels = torch.tensor(b["interaction"], dtype=torch.float32, device=DEVICE)
            loss = crit(preds, labels)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item(); nb += 1

        vm = metrics(val_ds.y, *collect(model, val_ds)[::-1]) if False else \
             metrics(*collect(model, val_ds))
        el = (datetime.datetime.now()-t0).total_seconds()
        eta = el/ep*(EPOCHS-ep)
        print(f"Epoch {ep:03d} | loss={ep_loss/nb:.4f} | val mcc={vm['mcc']:.3f} "
              f"acc={vm['acc']:.3f} aupr={vm['aupr']:.3f} | {el/60:.0f}min(eta {eta/60:.0f}min)")
        gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

        if vm["mcc"] > best_mcc:               # ★ 按 val mcc 选点（退化守卫）
            best_mcc, wait, best_ep = vm["mcc"], 0, ep
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"早停: 最优 val mcc={best_mcc:.3f} @ epoch {best_ep}")
                break

    # ---- 用最优 checkpoint 出终版数字（test 每种子只碰这一次）----
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    vm = metrics(*collect(model, val_ds))
    tm = metrics(*collect(model, test_ds))
    md5 = hashlib.md5(ckpt.read_bytes()).hexdigest()[:8]
    rec = {"seed":seed, "lr":LR, "frac":FRAC, "batch":BATCH, "best_epoch":best_ep,
           "val":vm, "test":tm, "ckpt":ckpt.name, "md5":md5,
           "ts":datetime.datetime.now().isoformat()}
    with open(RESULTS, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    print(f"\n[SEED {seed}] val aupr={vm['aupr']:.3f} | test aupr={tm['aupr']:.3f} "
          f"acc={tm['acc']:.3f} f1={tm['f1']:.3f} mcc={tm['mcc']:.3f} | md5={md5}")
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return rec

# ================= 聚合 =================
def aggregate():
    recs = [json.loads(l) for l in open(RESULTS, encoding="utf-8")]
    print(f"共 {len(recs)} 个种子: {[r['seed'] for r in recs]}\n")
    for r in recs:
        t = r["test"]
        print(f"  seed {r['seed']:>5}: test aupr={t['aupr']:.3f} acc={t['acc']:.3f} "
              f"f1={t['f1']:.3f} mcc={t['mcc']:.3f} (ep{r['best_epoch']}, {r['md5']})")
    def ms(key):
        v = np.array([r["test"][key] for r in recs])
        return v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0
    print(f"\n{'='*56}\n种子扫描汇总（n={len(recs)}）\n{'='*56}")
    for k in ("aupr","acc","f1","mcc"):
        m, s = ms(k)
        print(f"  test {k:>5} = {m:.3f} ± {s:.3f}")
    m, s = ms("aupr")
    print(f"\n  论文: aupr=0.641 | 本复现: {m:.3f}±{s:.3f} ({m-PAPER['aupr']:+.3f})")
    print("  → 差 |Δ|<2σ 即在种子噪声内; 报告写法: "
          f"'aupr = {m:.3f}±{s:.3f} (n={len(recs)} seeds) vs 0.641'")

# ================= 主流程 =================
def main():
    CKPT_DIR.mkdir(exist_ok=True)
    if len(sys.argv) > 1 and sys.argv[1] == "agg":
        aggregate(); return
    seeds = [int(a) for a in sys.argv[1:]] or [42]
    done = set()
    if RESULTS.exists():
        done = {json.loads(l)["seed"] for l in open(RESULTS, encoding="utf-8")}
        if done: print(f"已完成种子(跳过): {sorted(done)}")

    print("加载数据（一次，全部种子共用）…")
    train_ds = Split(load_split(TRAIN_POS, TRAIN_NEG))
    val_ds   = Split(load_split(VAL_POS,   VAL_NEG))
    test_ds  = Split(load_split(TEST_POS,  TEST_NEG))
    print(f"训练集样本/epoch = {int(len(train_ds)*FRAC)}（FRAC={FRAC}）\n")

    for seed in seeds:
        if seed in done: continue
        train_one_seed(seed, train_ds, val_ds, test_ds)
    print("\n全部种子完成 → python multiseed_train.py agg")
    if RESULTS.exists(): aggregate()

if __name__ == "__main__":
    main()


协议: LR=0.0001 FRAC=1.0 BATCH=16 PATIENCE=8 选点=val_mcc | Device=cuda



ValueError: invalid literal for int() with base 10: '--f=c:\\Users\\yuhao.bian\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3acc1c1037c4e2d2baee1e9e5a57001d3142d21c6.json'

In [11]:
# ranking_ceiling.py —— 排序上限 acc + 中段混合度（读 score_cache.npz, 秒级）
import numpy as np
from pathlib import Path

BASE = Path(r"D:/pythonprojects/practice-github/PPI_prediction(gold-standard dataset)")
z = np.load(BASE / "score_cache.npz")
pt, yt = z["pt"], z["yt"]

# ---- 排序上限: 在任意分数处切一刀, acc 的最大值 ----
ts = np.sort(np.unique(pt))
accs = np.array([((pt >= t) == yt).mean() for t in ts])
k = int(np.argmax(accs))
print(f"[排序上限] best acc = {accs[k]:.3f} @ 切分分位 "
      f"{(pt < ts[k]).mean():.2%}（0.5 分位 = 队列正中间）")
print(f"[对照]   acc@0.5 = {((pt>=0.5)==yt).mean():.3f} | 论文 acc = 0.616")
print(f"→ 校准空间 = 上限 - acc@0.5 = {accs[k] - ((pt>=0.5)==yt).mean():.3f}")
print(f"→ 排序传导 = 0.616 - 上限 = {0.616 - accs[k]:.3f}（acc 差距中来自排序的部分）\n")

# ---- 中段混合度: 十分位正率（0.5 = 随机）----
qs = np.quantile(pt, np.linspace(0, 1, 11))
print("[十分位正率] 队头→队尾（越接近 1/0 越纯, 中段 0.5 附近最混）")
for i in range(10):
    m = (pt >= qs[i]) & (pt <= qs[i+1])
    bar = "#" * int(yt[m].mean() * 20)
    print(f"  分位{i+1:2d}: 正率={yt[m].mean():.3f} {bar}")


[排序上限] best acc = 0.580 @ 切分分位 61.15%（0.5 分位 = 队列正中间）
[对照]   acc@0.5 = 0.578 | 论文 acc = 0.616
→ 校准空间 = 上限 - acc@0.5 = 0.002
→ 排序传导 = 0.616 - 上限 = 0.036（acc 差距中来自排序的部分）

[十分位正率] 队头→队尾（越接近 1/0 越纯, 中段 0.5 附近最混）
  分位 1: 正率=0.356 #######
  分位 2: 正率=0.404 ########
  分位 3: 正率=0.440 ########
  分位 4: 正率=0.468 #########
  分位 5: 正率=0.465 #########
  分位 6: 正率=0.486 #########
  分位 7: 正率=0.500 ##########
  分位 8: 正率=0.538 ##########
  分位 9: 正率=0.620 ############
  分位10: 正率=0.744 ##############
